# Unsupervised Learning — Tutorial

**Primary outcome:** Apply PCA for dimensionality reduction, t-SNE for visualization, and K-Means / Hierarchical / DBSCAN clustering on real datasets — and evaluate the quality of discovered structure.

**Estimated time:** 75–90 minutes

---

## What you will learn

| Section | Techniques |
|---------|------------|
| Part 0 | Dataset setup, scaling |
| Part 1 | PCA (scree plot, cumulative variance, biplots), t-SNE |
| Part 2 | K-Means (elbow, silhouette), Hierarchical (dendrogram), DBSCAN |
| Part 3 | Algorithm comparison, practice exercises |

> **No labels were harmed** — clustering discovers structure without using `y` during training.

## Part 0 — Setup

We import everything up front so later cells run independently.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import load_digits, make_moons, make_blobs
from sklearn.metrics import silhouette_score, adjusted_rand_score

from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded successfully.')

### 0.1 Load the Penguins dataset

We use the Palmer Penguins dataset for clustering tasks.
- Drop rows with missing values.
- Encode the `species` column as integers so we can use it to **evaluate** (not train) our clusters.

In [ ]:
# Load penguins from seaborn
penguins_raw = sns.load_dataset('penguins')
penguins = penguins_raw.dropna().reset_index(drop=True)

# Numeric features used for clustering
penguin_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X_penguins = penguins[penguin_features].values

# Encode species as integer labels (for evaluation only)
le = LabelEncoder()
y_penguins = le.fit_transform(penguins['species'])
species_names = le.classes_

# Scale features
scaler_penguins = StandardScaler()
X_penguins_scaled = scaler_penguins.fit_transform(X_penguins)

print(f'Penguins dataset: {X_penguins_scaled.shape[0]} samples, '
      f'{X_penguins_scaled.shape[1]} numeric features, '
      f'{len(species_names)} species')
print('Species:', list(species_names))

### 0.2 Load the Digits dataset

Each sample is an 8×8 grayscale image of a handwritten digit (0–9), flattened to 64 features.
We will use this high-dimensional dataset to demonstrate PCA and t-SNE.

In [ ]:
digits = load_digits()
X_digits = digits.data          # shape (1797, 64)
y_digits = digits.target        # 0-9

scaler_digits = StandardScaler()
X_digits_scaled = scaler_digits.fit_transform(X_digits)

print(f'Digits dataset: {X_digits_scaled.shape[0]} samples, '
      f'{X_digits_scaled.shape[1]} features, '
      f'{len(np.unique(y_digits))} digit classes')

# Quick peek at some digit images
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    idx = np.where(y_digits == digit)[0][0]
    axes[0, digit].imshow(digits.images[idx], cmap='gray_r')
    axes[0, digit].set_title(str(digit), fontsize=10)
    axes[0, digit].axis('off')
    idx2 = np.where(y_digits == digit)[0][1]
    axes[1, digit].imshow(digits.images[idx2], cmap='gray_r')
    axes[1, digit].axis('off')
plt.suptitle('Sample digits from sklearn.datasets.load_digits()', y=1.02)
plt.tight_layout()
plt.show()

---

## Part 1 — Dimensionality Reduction

### 1.1 Why Reduce Dimensions?

Real datasets often have tens, hundreds, or even thousands of features.  
Working directly in that high-dimensional space causes several problems:

**The Curse of Dimensionality**  
As dimensions grow, the volume of the space increases exponentially.  
Distance metrics become unreliable — everything looks equally far away — making clustering, classification, and nearest-neighbour search all harder.

**Why reduce?**
- Remove redundant or noisy features
- Visualise high-dimensional structure in 2D / 3D
- Speed up downstream models
- Reduce overfitting risk

**PCA vs t-SNE — when to use which?**

| Property | PCA | t-SNE |
|----------|-----|-------|
| Preserves | Global variance (linear) | Local neighbourhood structure (non-linear) |
| New data | Yes — `transform()` works on unseen samples | No — must refit from scratch |
| Interpretable components | Yes — loadings explain original features | No — axes have no meaning |
| Speed | Fast (O(n·d²)) | Slow (O(n²)) — use on ≤50k samples |
| Best for | Pre-processing before clustering/models | Final visualisation only |
| Key hyperparameter | `n_components` | `perplexity` (5–50, try 30) |

### 1.2 PCA on Digits — Scree Plot & Cumulative Variance

We first fit PCA with all 64 components to inspect how variance is distributed across components.

In [ ]:
# Fit full PCA to inspect variance distribution
pca_full = PCA(random_state=42)
pca_full.fit(X_digits_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)
n_95 = np.searchsorted(cumulative, 0.95) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot
axes[0].bar(range(1, 31), explained[:30], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot (first 30 components)')

# Cumulative explained variance
axes[1].plot(range(1, len(cumulative) + 1), cumulative, color='steelblue', linewidth=2)
axes[1].axhline(0.95, color='crimson', linestyle='--', label='95% threshold')
axes[1].axvline(n_95, color='orange', linestyle='--', label=f'{n_95} components')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance — Digits')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Components needed to explain 95% variance: {n_95}')
print(f'Variance explained by first 2 components: {cumulative[1]:.1%}')

**Reading the scree plot:** The first few components capture a steep drop in variance — keep components before the "elbow".

**What do principal components represent?**  
Each PC is a *linear combination* of the original features (pixel intensities in this case).  
PC1 points in the direction of maximum variance; PC2 is orthogonal to PC1 and captures the next most variance, and so on.  
Reducing from 64 → 41 dimensions retains 95% of the information — we lose very little while cutting nearly 40% of the features.

### 1.3 PCA 2D Scatter — Digits Coloured by True Label

In [ ]:
# Project digits to 2D with PCA
pca_2d = PCA(n_components=2, random_state=42)
X_digits_pca2 = pca_2d.fit_transform(X_digits_scaled)

palette = sns.color_palette('tab10', 10)
fig, ax = plt.subplots(figsize=(9, 7))

for digit in range(10):
    mask = y_digits == digit
    ax.scatter(X_digits_pca2[mask, 0], X_digits_pca2[mask, 1],
               label=str(digit), alpha=0.5, s=15, color=palette[digit])

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Digits projected onto first 2 PCA components')
ax.legend(title='Digit', bbox_to_anchor=(1.01, 1), loc='upper left', markerscale=2)
plt.tight_layout()
plt.show()

print(f'Variance captured in 2D: {pca_2d.explained_variance_ratio_.sum():.1%}')

### 1.4 PCA on Penguins — Biplot

With only 4 features, we can draw a **biplot**: data points in PC1/PC2 space plus arrows showing how each original feature contributes to those components.

In [ ]:
pca_penguins = PCA(n_components=2, random_state=42)
X_penguins_pca2 = pca_penguins.fit_transform(X_penguins_scaled)

species_colors = {'Adelie': '#E74C3C', 'Chinstrap': '#3498DB', 'Gentoo': '#2ECC71'}
colors = [species_colors[s] for s in penguins['species']]

fig, ax = plt.subplots(figsize=(9, 7))

# Scatter data points
for species, color in species_colors.items():
    mask = penguins['species'] == species
    ax.scatter(X_penguins_pca2[mask, 0], X_penguins_pca2[mask, 1],
               label=species, color=color, alpha=0.65, s=40)

# Draw feature loading arrows (biplot)
loadings = pca_penguins.components_.T  # shape (4, 2)
scale = 3.0
for i, feature in enumerate(penguin_features):
    ax.annotate('', xy=(loadings[i, 0] * scale, loadings[i, 1] * scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    ax.text(loadings[i, 0] * scale * 1.12, loadings[i, 1] * scale * 1.12,
            feature.replace('_mm', '').replace('_g', ''),
            fontsize=9, ha='center')

ax.set_xlabel(f'PC1 ({pca_penguins.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_penguins.explained_variance_ratio_[1]:.1%})')
ax.set_title('Penguin PCA Biplot — species separated without using labels')
ax.legend()
plt.tight_layout()
plt.show()

print('Loadings (feature contributions per PC):')
print(pd.DataFrame(pca_penguins.components_.T,
                   index=penguin_features,
                   columns=['PC1', 'PC2']).round(3))

**Key insight:** PCA separates species remarkably well using only the first two components — even though species labels were *never* used in fitting.  
The arrows show that `flipper_length_mm` and `body_mass_g` drive separation along PC1, while `bill_depth_mm` separates along PC2.

### 1.5 t-SNE Visualisation — Digits

t-SNE maps high-dimensional data to 2D by preserving **local neighbourhood structure**.  
It often produces visually cleaner clusters than PCA for complex data.

In [ ]:
# t-SNE on the full 64-dimensional digits (may take ~30 seconds)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_digits_tsne = tsne.fit_transform(X_digits_scaled)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, X_2d, title in zip(axes,
                            [X_digits_pca2, X_digits_tsne],
                            ['PCA (2 components)', 't-SNE (perplexity=30)']):
    for digit in range(10):
        mask = y_digits == digit
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   label=str(digit), alpha=0.5, s=12, color=palette[digit])
    ax.set_title(title)
    ax.legend(title='Digit', bbox_to_anchor=(1.01, 1), loc='upper left', markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('PCA vs t-SNE on Digits dataset', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**t-SNE vs PCA — key takeaways**

| | PCA | t-SNE |
|---|---|---|
| Cluster separation on digits | Good | Excellent |
| Can transform new data | Yes | **No** — must refit |
| Axes are interpretable | Yes (variance %) | **No** |
| Sensitive to `perplexity` | — | Yes — try 5, 30, 50 |
| Preserves global structure | Yes | Not reliably |

> **Rule of thumb:** Use PCA for pre-processing and feature engineering; use t-SNE only for final visualisation.

---

## Part 2 — Clustering

### 2.1 K-Means — Elbow Method & Silhouette Score

K-Means partitions data into *k* groups by minimising within-cluster sum of squared distances (inertia).  
We need to choose *k* before fitting. Two tools help:
- **Elbow plot:** Inertia drops steeply then levels off — the "elbow" is a good *k*.
- **Silhouette score:** Ranges from −1 to +1. Higher = tighter, better-separated clusters.

In [ ]:
inertias = []
silhouettes = []
k_range = range(2, 9)

for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_penguins_scaled)
    inertias.append(km.inertia_)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_penguins_scaled)
    silhouettes.append(silhouette_score(X_penguins_scaled, labels))

best_k = list(k_range)[np.argmax(silhouettes)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(range(1, 11), inertias, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('k (number of clusters)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method — Penguins')
axes[0].axvline(3, color='crimson', linestyle='--', label='k=3 (true species)')
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, 's-', color='darkorange', linewidth=2)
axes[1].set_xlabel('k (number of clusters)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — Penguins')
axes[1].axvline(best_k, color='crimson', linestyle='--', label=f'Best k={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Best k by silhouette score: {best_k} (score={max(silhouettes):.3f})')

### 2.2 K-Means with k=3 — Clusters vs True Species

In [ ]:
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
penguin_km_labels = km3.fit_predict(X_penguins_scaled)

ari = adjusted_rand_score(y_penguins, penguin_km_labels)

# Use 2D PCA projection for visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

titles = ['K-Means clusters (k=3)', 'True species']
label_sets = [penguin_km_labels, y_penguins]
label_names = [['Cluster 0', 'Cluster 1', 'Cluster 2'], list(species_names)]
cmap = ['#E74C3C', '#3498DB', '#2ECC71']

for ax, labels, names, title in zip(axes, label_sets, label_names, titles):
    for i, (label_val, name) in enumerate(zip(np.unique(labels), names)):
        mask = labels == label_val
        ax.scatter(X_penguins_pca2[mask, 0], X_penguins_pca2[mask, 1],
                   label=name, color=cmap[i], alpha=0.7, s=40)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title(title)
    ax.legend()

plt.suptitle(f'Adjusted Rand Index (K-Means vs True Labels): {ari:.3f}', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Adjusted Rand Index: {ari:.3f}  (1.0 = perfect match, 0.0 = random)')

### 2.3 K-Means Limitations — Non-Spherical Clusters

K-Means assumes clusters are roughly **spherical and equal-sized**.  
Apply it to `make_moons` data (two crescent shapes) to see where it fails.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
moons_km_labels = km_moons.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='bwr', s=20, alpha=0.8)
axes[0].set_title('True structure (two moons)')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=moons_km_labels, cmap='bwr', s=20, alpha=0.8)
axes[1].scatter(km_moons.cluster_centers_[:, 0], km_moons.cluster_centers_[:, 1],
                marker='X', s=200, color='black', zorder=5, label='Centroids')
axes[1].set_title('K-Means result — incorrect separation')
axes[1].legend()

plt.suptitle('K-Means fails on non-spherical clusters', fontsize=12)
plt.tight_layout()
plt.show()

print('K-Means ARI on moons:', round(adjusted_rand_score(y_moons, moons_km_labels), 3))
print('K-Means assumes spherical clusters and is sensitive to outliers.')

**Why K-Means struggles here:**  
The centroid-based update rule can only produce Voronoi-cell boundaries (straight lines in 2D).  
The crescent shapes require curved decision boundaries — K-Means simply cannot represent them.

### 2.4 Hierarchical Clustering — Dendrogram

Hierarchical clustering builds a **tree of merges** (agglomerative) or splits (divisive).  
The dendrogram shows which clusters merged at what distance, and lets us choose *k* by cutting the tree.

In [ ]:
# Use a subsample of 80 penguins for a readable dendrogram
np.random.seed(42)
idx80 = np.random.choice(len(X_penguins_scaled), 80, replace=False)
X_sub = X_penguins_scaled[idx80]
y_sub = y_penguins[idx80]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

linkage_methods = ['ward', 'complete', 'average']
for ax, method in zip(axes, linkage_methods):
    Z = linkage(X_sub, method=method)
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=20,
               leaf_rotation=90, leaf_font_size=8,
               color_threshold=Z[-3, 2])  # cut at 3 clusters
    ax.set_title(f'Linkage: {method}')
    ax.set_xlabel('Sample (or cluster size)')
    ax.set_ylabel('Distance')

plt.suptitle('Hierarchical Clustering Dendrograms — Penguins (80 samples)', fontsize=12)
plt.tight_layout()
plt.show()

### 2.5 Cut the Tree at 3 Clusters — Compare to K-Means and True Labels

In [ ]:
hc_ward = AgglomerativeClustering(n_clusters=3, linkage='ward')
penguin_hc_labels = hc_ward.fit_predict(X_penguins_scaled)

hc_ari = adjusted_rand_score(y_penguins, penguin_hc_labels)
hc_sil = silhouette_score(X_penguins_scaled, penguin_hc_labels)
km_sil = silhouette_score(X_penguins_scaled, penguin_km_labels)

print('--- Penguins: Clustering comparison ---')
print(f'K-Means (k=3):           ARI={ari:.3f}  Silhouette={km_sil:.3f}')
print(f'Hierarchical (ward, k=3): ARI={hc_ari:.3f}  Silhouette={hc_sil:.3f}')

# Scatter: hierarchical clusters vs true species
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, labels, title in zip(axes,
                               [penguin_hc_labels, y_penguins],
                               ['Hierarchical (Ward, k=3)', 'True species']):
    for i in range(3):
        mask = labels == i
        ax.scatter(X_penguins_pca2[mask, 0], X_penguins_pca2[mask, 1],
                   color=cmap[i], alpha=0.7, s=40, label=str(i))
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title(title); ax.legend()
plt.tight_layout()
plt.show()

### 2.6 DBSCAN — Density-Based Clustering

DBSCAN groups points that are **densely packed together** and marks sparse points as **noise** (label = −1).  
Two parameters control the algorithm:
- **`eps`** — maximum distance between two points to be considered neighbours.
- **`min_samples`** — minimum number of neighbours required to form a core point.

DBSCAN can discover **arbitrarily-shaped clusters** and handles outliers natively.

In [ ]:
# DBSCAN on make_moons — where K-Means failed
eps_values = [0.1, 0.2, 0.3, 0.5]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, eps in zip(axes, eps_values):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)

    # Noise points in grey
    noise_mask = labels == -1
    ax.scatter(X_moons[noise_mask, 0], X_moons[noise_mask, 1],
               c='lightgrey', s=20, label='Noise', zorder=2)
    ax.scatter(X_moons[~noise_mask, 0], X_moons[~noise_mask, 1],
               c=labels[~noise_mask], cmap='bwr', s=20, alpha=0.8)

    ax.set_title(f'eps={eps}\n{n_clusters} cluster(s), {n_noise} noise')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('DBSCAN on make_moons — varying eps (min_samples=5)', fontsize=12)
plt.tight_layout()
plt.show()

print('eps=0.2 correctly recovers the two moon shapes.')

### 2.7 DBSCAN on Penguins

In [ ]:
db_penguins = DBSCAN(eps=1.2, min_samples=8)
penguin_db_labels = db_penguins.fit_predict(X_penguins_scaled)

n_clusters_db = len(set(penguin_db_labels)) - (1 if -1 in penguin_db_labels else 0)
n_noise_db = np.sum(penguin_db_labels == -1)

print(f'DBSCAN found {n_clusters_db} cluster(s) and {n_noise_db} noise point(s)')

if n_clusters_db > 1:
    valid = penguin_db_labels != -1
    db_sil = silhouette_score(X_penguins_scaled[valid], penguin_db_labels[valid])
    db_ari = adjusted_rand_score(y_penguins, penguin_db_labels)
    print(f'Silhouette (non-noise): {db_sil:.3f}')
    print(f'ARI vs true species:    {db_ari:.3f}')
else:
    print('Only 1 cluster found — try reducing eps or min_samples.')

# Visualise
fig, ax = plt.subplots(figsize=(7, 5))
unique_labels = sorted(set(penguin_db_labels))
colors_db = ['lightgrey' if l == -1 else cmap[l % 3] for l in unique_labels]
for l, c in zip(unique_labels, colors_db):
    mask = penguin_db_labels == l
    label_name = 'Noise' if l == -1 else f'Cluster {l}'
    ax.scatter(X_penguins_pca2[mask, 0], X_penguins_pca2[mask, 1],
               color=c, s=40, alpha=0.75, label=label_name)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'DBSCAN on Penguins (eps=1.2, min_samples=8)')
ax.legend()
plt.tight_layout()
plt.show()

---

## Part 3 — Choosing the Right Algorithm

### 3.1 Algorithm Comparison Table

| Property | K-Means | Hierarchical | DBSCAN |
|----------|---------|--------------|--------|
| Must specify k | **Yes** | Yes (or cut height) | No |
| Cluster shape | Spherical only | Flexible (depends on linkage) | **Arbitrary** |
| Handles noise | No | No | **Yes** (marks as -1) |
| Scalability | **Fast** O(nkd) | Slow O(n²) | Moderate O(n log n) with index |
| Sensitive to scale | Yes | Yes | **Yes** — always scale first |
| Deterministic | No (random init) | **Yes** | **Yes** |
| Best for | Large, compact clusters | Hierarchical structure, dendrograms | Arbitrary shapes, noisy data |

### 3.2 Head-to-Head — Apply All Three to Both Datasets

In [ ]:
from sklearn.preprocessing import StandardScaler as SS

X_moons_scaled = SS().fit_transform(X_moons)

results = []

datasets = {
    'Penguins': (X_penguins_scaled, y_penguins),
    'Moons':    (X_moons_scaled,    y_moons)
}

algorithms = {
    'K-Means':      lambda X, y: KMeans(n_clusters=len(np.unique(y)), random_state=42, n_init=10).fit_predict(X),
    'Hierarchical': lambda X, y: AgglomerativeClustering(n_clusters=len(np.unique(y)), linkage='ward').fit_predict(X),
    'DBSCAN':       lambda X, y: DBSCAN(eps=0.4 if X.shape[1] == 2 else 1.2,
                                         min_samples=5 if X.shape[1] == 2 else 8).fit_predict(X)
}

for ds_name, (X, y) in datasets.items():
    for alg_name, fit_fn in algorithms.items():
        labels = fit_fn(X, y)
        n_found = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int(np.sum(labels == -1))
        valid = labels != -1
        sil = silhouette_score(X[valid], labels[valid]) if n_found > 1 and valid.sum() > 1 else float('nan')
        ari = adjusted_rand_score(y, labels)
        results.append({'Dataset': ds_name, 'Algorithm': alg_name,
                        'Clusters found': n_found, 'Noise pts': n_noise,
                        'Silhouette': round(sil, 3), 'ARI': round(ari, 3)})

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

---

## Part 4 — Practice Exercises

Work through each exercise independently.  
Expand the **Solution** cell when you're ready to check your work.

### Exercise 1 — Most Important Penguin Feature for PCA

Apply PCA to the scaled penguin dataset and identify which of the 4 features contributes **most** to PC1.  
Print the feature name and its loading magnitude.

**Hint:** Look at `pca.components_[0]` — it is the loading vector for PC1.

In [ ]:
# Your code here


<details>
<summary><strong>Solution — Exercise 1</strong></summary>

```python
pca_ex1 = PCA(n_components=4, random_state=42)
pca_ex1.fit(X_penguins_scaled)

loadings_pc1 = pca_ex1.components_[0]  # PC1 loading vector
abs_loadings = np.abs(loadings_pc1)
most_important_idx = np.argmax(abs_loadings)
most_important_feature = penguin_features[most_important_idx]

print('PC1 loadings:')
for feat, load in zip(penguin_features, loadings_pc1):
    print(f'  {feat:25s}: {load:+.3f}')
print(f'\nMost important feature for PC1: {most_important_feature} (|loading|={abs_loadings[most_important_idx]:.3f})')
```

Expected output: `flipper_length_mm` has the highest absolute loading on PC1, meaning it drives the first principal component most strongly.
</details>

### Exercise 2 — Optimal k for the Digits Dataset

Use K-Means with the **silhouette score** to find the best number of clusters for `X_digits_scaled`.  
Test k = 5, 8, 10, 12, 15 and print the best k and its score.

**Hint:** Reduce computation — first apply PCA to keep 50 components, then run K-Means.

In [ ]:
# Your code here


<details>
<summary><strong>Solution — Exercise 2</strong></summary>

```python
# Reduce dims first for speed
pca_50 = PCA(n_components=50, random_state=42)
X_digits_50 = pca_50.fit_transform(X_digits_scaled)

k_candidates = [5, 8, 10, 12, 15]
scores = []

for k in k_candidates:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_digits_50)
    scores.append(silhouette_score(X_digits_50, labels))
    print(f'k={k:2d}  silhouette={scores[-1]:.4f}')

best_idx = np.argmax(scores)
print(f'\nBest k={k_candidates[best_idx]} (silhouette={max(scores):.4f})')
```

You should find k=10 achieves a strong silhouette score, matching the true number of digit classes.
</details>

### Exercise 3 — Tune DBSCAN eps on the Moons Dataset

Systematically search `eps` in `[0.05, 0.1, 0.15, 0.2, 0.25, 0.3]` with `min_samples=5` on `X_moons`.  
For each eps, print: number of clusters found, number of noise points, silhouette score (if >1 cluster).  
Identify the best eps that gives exactly 2 clusters with the highest silhouette score.

In [ ]:
# Your code here


<details>
<summary><strong>Solution — Exercise 3</strong></summary>

```python
eps_candidates = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
best_eps, best_sil = None, -1

for eps in eps_candidates:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_clust = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int(np.sum(labels == -1))
    valid = labels != -1
    if n_clust > 1 and valid.sum() > 1:
        sil = silhouette_score(X_moons[valid], labels[valid])
        if n_clust == 2 and sil > best_sil:
            best_sil, best_eps = sil, eps
    else:
        sil = float('nan')
    print(f'eps={eps:.2f}  clusters={n_clust}  noise={n_noise}  silhouette={sil:.3f}' if not np.isnan(sil) else
          f'eps={eps:.2f}  clusters={n_clust}  noise={n_noise}  silhouette=N/A')

print(f'\nBest eps for 2 clusters: {best_eps} (silhouette={best_sil:.3f})')
```

`eps=0.20` typically gives 2 clean clusters with a high silhouette score and minimal noise.
</details>

---

## Summary

| Technique | When to use | Key parameter |
|-----------|-------------|---------------|
| **PCA** | Pre-processing, noise reduction, feature engineering | `n_components` or variance threshold |
| **t-SNE** | Final 2D/3D visualisation only | `perplexity` (try 5–50) |
| **K-Means** | Large datasets, compact spherical clusters | `n_clusters` (use elbow + silhouette) |
| **Hierarchical** | When you need a tree structure; small-medium data | linkage method, cut height |
| **DBSCAN** | Irregular shapes, noise/outliers present | `eps`, `min_samples` |

**Evaluation metrics**
- **Silhouette score** — internal measure (no labels needed). Closer to +1 is better.
- **Adjusted Rand Index** — external measure (requires true labels). 1.0 = perfect, 0.0 = random.

**General pipeline**
1. Scale features with `StandardScaler`.
2. Optionally reduce with PCA (keeps 95% variance).
3. Try multiple algorithms; compare silhouette scores.
4. Visualise with PCA or t-SNE.
5. If true labels exist, compute ARI to validate.